# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbtissamToure/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule (plain words):**

Prioritize content that has not been updated for at least 180 days but still receives at least 500 impressions over the last 90 days. Rank qualifying content by impressions, with higher-impression pages reviewed first, because refreshing older content that still attracts search visibility may offer a recovery opportunity.

**Reason code:** `stale_but_visible`

**Action label:** `review_refresh`

**Hypothesis to test:** Staleness and search visibility may help identify content worth reviewing for recovery. I will check both signals using bucket tables before deciding whether to keep or revise this rule.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

# Find the project root from the notebook's current location
current = Path.cwd()
project_root = next(
    (p for p in [current, *current.parents]
     if (p / "data/raw/content_refresh_anonymized.csv").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv"
    )

data_path = project_root / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print("\nRequired columns:")

required_columns = [
    "days_since_last_update",
    "impressions_90d",
]

for column in required_columns:
    print(f"{column}: {'Found' if column in df.columns else 'MISSING'}")

df[required_columns].describe()

Rows: 30,000
Columns: 44

Required columns:
days_since_last_update: Found
impressions_90d: Found


,days_since_last_update,impressions_90d
count,30000.000000,30000.000000
mean,46.098300,5200.366300
std,42.078709,16838.019547
min,1.000000,1.000000
25%,20.000000,81.000000
50%,20.000000,731.000000
75%,104.000000,3615.250000
max,373.000000,517715.000000


In [7]:

# Signal 1: Content staleness
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, float("inf")],
    labels=["1–30 days", "31–90 days", "91–180 days", "181+ days"],
    include_lowest=True
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
    .agg(
        n=("days_since_last_update", "size"),
        median_impressions=("impressions_90d", "median"),
        pct_with_500_plus_impressions=(
            "impressions_90d",
            lambda x: (x >= 500).mean() * 100
        )
    )
    .reset_index()
)

print("Signal 1: Staleness")
display(staleness_table.round(2))


# Signal 2: Search visibility (impressions)
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 99, 499, 2999, float("inf")],
    labels=["1–99", "100–499", "500–2,999", "3,000+"],
    include_lowest=True
)

volume_table = (
    df.groupby("volume_bucket", observed=False)
    .agg(
        n=("impressions_90d", "size"),
        median_days_since_update=("days_since_last_update", "median"),
        pct_stale_180_plus=(
            "days_since_last_update",
            lambda x: (x >= 180).mean() * 100
        )
    )
    .reset_index()
)

print("\nSignal 2: Search visibility")
display(volume_table.round(2))


Signal 1: Staleness


,staleness_bucket,n,median_impressions,pct_with_500_plus_impressions
0,1–30 days,20480,470.0,49.14
1,31–90 days,175,510.0,50.29
2,91–180 days,9171,1692.0,71.51
3,181+ days,174,15.5,9.77



Signal 2: Search visibility


,volume_bucket,n,median_days_since_update,pct_stale_180_plus
0,1–99,7994,20.0,1.74
1,100–499,5280,22.0,0.34
2,"500–2,999",8443,22.0,0.09
3,"3,000+",8283,25.0,0.11


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Rule (plain words):**

Prioritize content with at least 3,000 impressions over the last 90 days for review, ranking pages by impressions from highest to lowest. These pages have demonstrated search visibility and may offer opportunities to investigate growth or recovery.

**Reason code:** `high_search_visibility`

**Action label:** `review_growth_opportunity`

This is a transparent baseline for prioritizing review, not a validated prediction of future decline.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Build the ranked queue using one transparent rule
threshold = 3000

queue = df.loc[
    df["impressions_90d"] >= threshold
].copy()

queue["score"] = queue["impressions_90d"]
queue["reason_code"] = "high_search_visibility"
queue["action_label"] = "review_growth_opportunity"

queue = queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

# Keep the output focused on the action queue
output_columns = [
    "content_id",
    "score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
]

queue = queue[output_columns]

output_dir = project_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print(f"Qualifying pages: {len(queue):,}")
print(f"Saved ranked queue to: {output_path}")

display(queue.head(10))


Qualifying pages: 8,283
Saved ranked queue to: /Users/ibtissamtoure/FlyRank-Internship/work/outputs/baseline_action_score.csv


,content_id,score,reason_code,action_label,impressions_90d,days_since_last_update
0,content_5fe46e04994d,517715,high_search_visibility,review_growth_opportunity,517715,104
1,content_aaef01a50def,517109,high_search_visibility,review_growth_opportunity,517109,22
2,content_8c19996aa890,509252,high_search_visibility,review_growth_opportunity,509252,20
3,content_2cb567c3c89b,497727,high_search_visibility,review_growth_opportunity,497727,48
4,content_4c36c775b818,463103,high_search_visibility,review_growth_opportunity,463103,20
5,content_2dba2b1f9536,443434,high_search_visibility,review_growth_opportunity,443434,104
6,content_1a9e894be2e2,416180,high_search_visibility,review_growth_opportunity,416180,22
7,content_2c2606c5d176,347399,high_search_visibility,review_growth_opportunity,347399,104
8,content_db5989a78dd3,345111,high_search_visibility,review_growth_opportunity,345111,20
9,content_44e481c8f55b,312694,high_search_visibility,review_growth_opportunity,312694,20


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Review the top 10 ranked pages
top10 = queue.head(10).copy()

for _, row in top10.iterrows():
    print(
        f"{row['content_id']} | "
        f"Action: {row['action_label']} | "
        f"Why: {row['impressions_90d']:,} impressions in 90 days "
        f"(score: {row['score']:,}) | "
        f"What could make this wrong: High impressions alone "
        f"do not prove a growth opportunity; the page may already "
        f"be performing well or have little room to improve."
    )


content_5fe46e04994d | Action: review_growth_opportunity | Why: 517,715 impressions in 90 days (score: 517,715) | What could make this wrong: High impressions alone do not prove a growth opportunity; the page may already be performing well or have little room to improve.
content_aaef01a50def | Action: review_growth_opportunity | Why: 517,109 impressions in 90 days (score: 517,109) | What could make this wrong: High impressions alone do not prove a growth opportunity; the page may already be performing well or have little room to improve.
content_8c19996aa890 | Action: review_growth_opportunity | Why: 509,252 impressions in 90 days (score: 509,252) | What could make this wrong: High impressions alone do not prove a growth opportunity; the page may already be performing well or have little room to improve.
content_2cb567c3c89b | Action: review_growth_opportunity | Why: 497,727 impressions in 90 days (score: 497,727) | What could make this wrong: High impressions alone do not prove a grow

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Inspect the 5 weakest pages that still qualify for the queue
weak_picks = queue.tail(5)

print("Weak picks: lowest-scoring pages that still meet the rule")
display(
    weak_picks[
        [
            "content_id",
            "score",
            "impressions_90d",
            "days_since_last_update",
            "action_label",
            "reason_code",
        ]
    ]
)

# Leakage check: ensure label-derived trend fields are not used
# in the rule or included as inputs in the ranked queue.
forbidden_inputs = {"trend_direction", "trend_pct", "is_declining_label"}
rule_inputs = {"impressions_90d"}

leaked_inputs = rule_inputs & forbidden_inputs
queue_leakage = forbidden_inputs.intersection(queue.columns)

print("\nLeakage check:")
print(f"Rule uses only impressions_90d: {not leaked_inputs}")
print(f"No label-derived trend fields in queue: {not queue_leakage}")

assert not leaked_inputs, f"Forbidden rule inputs: {leaked_inputs}"
assert not queue_leakage, f"Label-derived fields in queue: {queue_leakage}"


Weak picks: lowest-scoring pages that still meet the rule


,content_id,score,impressions_90d,days_since_last_update,action_label,reason_code
8278,content_8b7ca0db5586,3003,3003,22,review_growth_opportunity,high_search_visibility
8279,content_f52dca1410d5,3003,3003,20,review_growth_opportunity,high_search_visibility
8280,content_40b82b1becdb,3001,3001,8,review_growth_opportunity,high_search_visibility
8281,content_5eb2f48aab0f,3001,3001,7,review_growth_opportunity,high_search_visibility
8282,content_747a3232517a,3001,3001,22,review_growth_opportunity,high_search_visibility



Leakage check:
Rule uses only impressions_90d: True
No label-derived trend fields in queue: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.